# Metadata Filtering & Hybrid Search
### Two upgrades on top of Naive RAG (see `Simple_RAG.ipynb`)

## Step 0: Recap — rebuild the same pipeline as Simple RAG
PDF → Chunks → Embeddings → FAISS

In [1]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters langchain-classic faiss-cpu pypdf rank_bm25 -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from collections import Counter

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

C:\Users\shiva\AppData\Local\Temp\ipykernel_5392\1759514193.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Pages: {len(pages)}")
print(f"Total Chunks: {len(chunks)}")

Total Pages: 30
Total Chunks: 136


## Step 1: Add metadata to each chunk
Real documents usually carry metadata — department, author, date, doc type. This PDF doesn't, so we'll simulate it using the article's own structure: the page a chunk came from tells us which *section* of the biography it belongs to, and lets us flag the citation/footnote pages as noise.

In [4]:
def tag_section(page: int) -> str:
    if page <= 1:
        return "Early Life"
    elif page <= 3:
        return "Career & Scientific Work"
    elif page <= 5:
        return "Presidency & Post-Presidency"
    elif page <= 9:
        return "Personal Life, Legacy & Awards"
    else:
        return "References"  # citation/footnote pages - not useful prose

for chunk in chunks:
    chunk.metadata["section"] = tag_section(chunk.metadata["page"])

print(Counter(chunk.metadata["section"] for chunk in chunks))

Counter({'References': 98, 'Personal Life, Legacy & Awards': 16, 'Career & Scientific Work': 9, 'Presidency & Post-Presidency': 7, 'Early Life': 6})


In [5]:
# a chunk now carries a 'section' tag alongside the usual page/source metadata
print(chunks[10].metadata)

{'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-03-09T06:22:40+00:00', 'title': 'A. P. J. Abdul Kalam - Wikipedia', 'moddate': '2026-03-09T06:22:40+00:00', 'source': 'A._P._J._Abdul_Kalam.pdf', 'total_pages': 30, 'page': 2, 'page_label': '3', 'section': 'Career & Scientific Work'}


## Step 2: Embed & store in FAISS
(identical to Simple RAG — the metadata just rides along with each chunk)

In [6]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


---
## Part A: Metadata Filtering

Metadata filtering narrows the search space with exact conditions on a document's metadata — applied alongside the similarity search, not instead of it. Useful for excluding noisy content, restricting to one category, date ranges, permissions, etc.

FAISS's `filter` argument accepts a plain dict (exact match), or operators like `$neq`, `$in`, `$and`, `$or`.

### Demo 1 — exclude a noisy section
Ask about awards. Without a filter, a citation/footnote chunk from the References pages can sneak into the top results.

In [7]:
query = "What awards did Abdul Kalam receive?"

print("Without filter:")
for doc in vector_store.similarity_search(query, k=4):
    print(f"[{doc.metadata['section']}] page {doc.metadata['page']}: {doc.page_content[:120]}...")

Without filter:
[References] page 25: 148. "Bharat Ratna conferred on Dr Abdul Kalam" (http://www.rediff.com/news/nov/26kal.htm).
Rediff.com. 26 November 1997...
[Personal Life, Legacy & Awards] page 9: of Tamil Nadu announced that Kalam's birthday, 15 October, would be observed as "Youth Renaissance
Day". It also institu...
[Early Life] page 0: Party and the then-opposition Indian National
Congress. He was widely referred to as the "People's
President". He engage...
[Personal Life, Legacy & Awards] page 9: and modernisation of defence technology in India.[148] He received the Indira Gandhi Award for National
Integration in 1...


In [8]:
print("With filter (exclude References):")
for doc in vector_store.similarity_search(query, k=4, filter={"section": {"$neq": "References"}}):
    print(f"[{doc.metadata['section']}] page {doc.metadata['page']}: {doc.page_content[:120]}...")

With filter (exclude References):
[Personal Life, Legacy & Awards] page 9: of Tamil Nadu announced that Kalam's birthday, 15 October, would be observed as "Youth Renaissance
Day". It also institu...
[Early Life] page 0: Party and the then-opposition Indian National
Congress. He was widely referred to as the "People's
President". He engage...
[Personal Life, Legacy & Awards] page 9: and modernisation of defence technology in India.[148] He received the Indira Gandhi Award for National
Integration in 1...
[Personal Life, Legacy & Awards] page 9: Books. Rupa Publications. ISBN 978-8-129-13486-8.
A. P. J. Abdul Kalam; Srijan Pal Singh (2015). Reignited: Scientific P...


### Demo 2 — restrict to one section
Ask specifically about his scientific career, filtered to just the `Career & Scientific Work` section.

Note: FAISS applies the filter *after* fetching `fetch_k` nearest candidates — widen `fetch_k` when a filter is narrow, or you may get back fewer than `k` results.

In [9]:
query = "What missile programs and DRDO work did Kalam lead?"

print("Without filter:")
for doc in vector_store.similarity_search(query, k=4):
    print(f"[{doc.metadata['section']}] page {doc.metadata['page']}: {doc.page_content[:120]}...")

Without filter:
[Career & Scientific Work] page 3: US$780 million in 2023) for the project titled Integrated Guided Missile Development Programme
(IGMDP) and appointed Kal...
[Career & Scientific Work] page 3: Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true...
[Early Life] page 0: 2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineer...
[References] page 14: realised-Nag-missile-dream-to-become-reality-next-year/articleshow/48267342.cms) from
the original on 3 January 2017. Re...


In [10]:
print("Filtered to 'Career & Scientific Work':")
for doc in vector_store.similarity_search(query, k=4, filter={"section": "Career & Scientific Work"}, fetch_k=50):
    print(f"[{doc.metadata['section']}] page {doc.metadata['page']}: {doc.page_content[:120]}...")

Filtered to 'Career & Scientific Work':
[Career & Scientific Work] page 3: US$780 million in 2023) for the project titled Integrated Guided Missile Development Programme
(IGMDP) and appointed Kal...
[Career & Scientific Work] page 3: Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true...
[Career & Scientific Work] page 2: received the approval from the Government of India to expand the programme to include more
engineers.[19] In 1963–64, he...
[Career & Scientific Work] page 2: career, he was involved in the design of small
hovercraft, and remained unconvinced by his choice
of a job at DRDO.[20] ...


---
## Part B: Hybrid Search

Vector search matches *meaning* but can blur exact names and numbers. Keyword search (BM25) matches *exact terms* but misses paraphrases. Hybrid search runs both and merges the rankings — this is what most production RAG systems use.

In [11]:
narrative_chunks = [c for c in chunks if c.metadata["section"] != "References"]

bm25_retriever = BM25Retriever.from_documents(narrative_chunks)
bm25_retriever.k = 3

vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3, "filter": {"section": {"$neq": "References"}}}
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

### Demo 1 — a query with an exact named entity
`"Bharat Ratna"` is a proper noun — an easy win for keyword search, but pure vector search can blur it with other award/book chunks.

In [12]:
query = "Bharat Ratna"

print("BM25 (keyword) only:")
for doc in bm25_retriever.invoke(query):
    print(f"page {doc.metadata['page']}: {doc.page_content[:100]}...")

print("\nVector (semantic) only:")
for doc in vector_retriever.invoke(query):
    print(f"page {doc.metadata['page']}: {doc.page_content[:100]}...")

print("\nHybrid:")
for doc in hybrid_retriever.invoke(query):
    print(f"page {doc.metadata['page']}: {doc.page_content[:100]}...")

BM25 (keyword) only:
page 1: Preceded by Raja Ramanna
Succeeded by Vasudev Kalkunte Aatre
Personal details
Born 15 October 1931
R...
page 0: Party and the then-opposition Indian National
Congress. He was widely referred to as the "People's
P...
page 9: Books. Rupa Publications. ISBN 978-8-129-13486-8.
A. P. J. Abdul Kalam; Srijan Pal Singh (2015). Rei...

Vector (semantic) only:
page 8: A. P. J. Abdul Kalam; Poonam Kohli (2012). You are Unique: Scale New Heights by
Thoughts and Actions...
page 8: Ocean Books. ISBN 978-8-188-32274-9.
A. P. J. Abdul Kalam; Manav Gupta (2005). Mission India : A Vis...
page 8: Space Technology. Indian Academy of Sciences.[143]
A. P. J. Abdul Kalam; Y. S. Rajan (1998). India 2...

Hybrid:
page 1: Preceded by Raja Ramanna
Succeeded by Vasudev Kalkunte Aatre
Personal details
Born 15 October 1931
R...
page 8: A. P. J. Abdul Kalam; Poonam Kohli (2012). You are Unique: Scale New Heights by
Thoughts and Actions...
page 0: Party and the then-opposition Indian Nati

### Demo 2 — a paraphrased query with no shared keywords
Now flip it: a natural-language question that doesn't share exact wording with the source. Vector search should do better here.

In [13]:
query = "Where was Kalam born?"

print("BM25 (keyword) only:")
for doc in bm25_retriever.invoke(query):
    print(f"page {doc.metadata['page']}: {doc.page_content[:100]}...")

print("\nVector (semantic) only:")
for doc in vector_retriever.invoke(query):
    print(f"page {doc.metadata['page']}: {doc.page_content[:100]}...")

print("\nHybrid:")
for doc in hybrid_retriever.invoke(query):
    print(f"page {doc.metadata['page']}: {doc.page_content[:100]}...")

BM25 (keyword) only:
page 6: Kalam's veena on display at the
Rashtrapati Bhavan museum in Delhi
until 8 p.m. that evening.[104][1...
page 6: Kalam was the youngest of five siblings, the eldest of whom
was a sister, Asim Zohra (d. 1997), foll...
page 2: This was my first stage, in which I learnt
leadership from three great teachers—Dr
Vikram Sarabhai, ...

Vector (semantic) only:
page 1: Organisation
Website A. P. J. Abdul Kalam Centre (h
ttps://kalamcentre.com)
Signature
Kalam's birthp...
page 9: of Tamil Nadu announced that Kalam's birthday, 15 October, would be observed as "Youth Renaissance
D...
page 1: Preceded by Raja Ramanna
Succeeded by Vasudev Kalkunte Aatre
Personal details
Born 15 October 1931
R...

Hybrid:


page 6: Kalam's veena on display at the
Rashtrapati Bhavan museum in Delhi
until 8 p.m. that evening.[104][1...
page 1: Organisation
Website A. P. J. Abdul Kalam Centre (h
ttps://kalamcentre.com)
Signature
Kalam's birthp...
page 6: Kalam was the youngest of five siblings, the eldest of whom
was a sister, Asim Zohra (d. 1997), foll...
page 9: of Tamil Nadu announced that Kalam's birthday, 15 October, would be observed as "Youth Renaissance
D...
page 2: This was my first stage, in which I learnt
leadership from three great teachers—Dr
Vikram Sarabhai, ...
page 1: Preceded by Raja Ramanna
Succeeded by Vasudev Kalkunte Aatre
Personal details
Born 15 October 1931
R...


Neither retriever wins every time — that's the point. Hybrid search hedges across both failure modes instead of betting on one.

---
## Putting it together: filtered + hybrid retrieval → LLM

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

bm25_retriever = BM25Retriever.from_documents(narrative_chunks)
bm25_retriever.k = 3

vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3, "filter": {"section": {"$neq": "References"}}}
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)


llm = ChatOllama(model="llama3.2:3b", temperature=0)

query = "What awards did Abdul Kalam receive?"
retrieved_docs = hybrid_retriever.invoke(query)
context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

response = llm.invoke(prompt)
print("Answer:", response.content)

Answer: The text mentions the following awards received by Abdul Kalam:

1. Padma Bhushan (1981)
2. Padma Vibhushan (1990)
3. Bharat Ratna (1997)

Additionally, he received several other awards, including:

* Indira Gandhi Award for National Integration
* Savarkar Award
* Ramanujan Award
* Hoover Medal
* Von Braun Award


## Try it yourself
1. Filter to `"Presidency & Post-Presidency"` and ask an election-related question.
2. Change the hybrid weights to `[0.8, 0.2]` (favor keyword) and see how results shift.
3. Combine filters with `$and`, e.g. `section` plus a page-number condition.